# Nismara Smart Analytics: Laporan Perbandingan (Month vs Month)
Versi diperluas: Meliputi data Logistik (ETS2 vs ATS, Singleplayer vs TruckersMP), Aktivitas Komunitas (Convoy), Kontrak Khusus, Event Boost, Penggunaan Kupon, Riwayat Pembayaran Penalti, Hardcore Mode, XP, dan Diskon.

In [ ]:
!pip install mcp python-dotenv pandas google-generativeai

In [ ]:
import asyncio
import os
import json
import pandas as pd
import google.generativeai as genai
from dotenv import load_dotenv
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel('gemini-3.5-flash')

server_params = StdioServerParameters(
    command="node",
    args=["build/index.js"], 
    env={
        **os.environ,
        "MONGODB_URI": os.getenv("MONGODB_URI"),
        "MCP_API_TOKEN": os.getenv("MCP_API_TOKEN")
    }
)

### TAHAP 1: Ekspor Multi-Koleksi
Menarik data dari 6 koleksi berbeda untuk DUA bulan sekaligus dan menyimpannya sebagai file cache.

In [ ]:
async def fetch_collection(session, collection_name, query_json):
    result = await session.call_tool("query_collection", arguments={
        "collectionName": collection_name, 
        "query": query_json,
        "limit": 5000,
        "sort": '{"_id": -1}'
    })
    data_mentah = result.content[0].text
    return [] if data_mentah == "[]" else json.loads(data_mentah)

async def export_all_data(bulan, tahun, gte_date, lte_date):
    async with stdio_client(server_params, errlog=None) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            print(f"\n🔍 Menarik data {bulan} {tahun} dari seluruh ekosistem...")
            
            query_json = f'{{ "createdAt": {{ "$gte": "{gte_date}", "$lte": "{lte_date}" }} }}'
            koleksi_utama = ["jobhistories", "convoylobby", "coupons", "contracthistories", "ncevents", "pointhistories"]
            
            for col in koleksi_utama:
                data = await fetch_collection(session, col, query_json)
                filename = f"data_{col}_{bulan.lower()}{tahun}.json"
                
                if col == "jobhistories":
                    if not data:
                        print(f" ├── [jobhistories] 0 data ditemukan.")
                        continue
                    df = pd.json_normalize(data)
                    # PENAMBAHAN FIELD BERDASARKAN SCHEMA TERBARU
                    kolom_penting = [
                        'jobId', 'jobStatus', 'game', 'gameMode', 'isHardcore', 'hardcoreRating', 'marketType',
                        'createdAt', 'cargoName', 'cargoMass', 'vehicleStatus',
                        'sourceCity', 'destinationCity', 'distanceKm', 'durationSeconds', 
                        'nc.total', 'nc.base', 'nc.special', 'nc.hardcore', 'nc.event', 'nc.booster',
                        'xp.total', 'discount.total',
                        'damage.vehicle', 'damage.trailer', 'damage.cargo',
                        'penalty.vehicle', 'penalty.trailer', 'penalty.cargo', 
                        'penalty.speed', 'penalty.distance', 'penalty.total',
                        'ncCost.rent', 'ncCost.service', 'ncCost.fuel', 'ncCost.fines', 'ncCost.total',
                        'vehicle.brand', 'vehicle.model'
                    ]
                    kolom_tersedia = [c for c in kolom_penting if c in df.columns]
                    df[kolom_tersedia].to_csv(filename.replace(".json", ".csv"), index=False, encoding="utf-8-sig")
                    print(f" ├── [{col}] {len(data)} data diekspor ke CSV bersih.")
                else:
                    with open(filename, "w", encoding="utf-8") as f:
                        json.dump(data, f, ensure_ascii=False, indent=2)
                    print(f" ├── [{col}] {len(data)} data diekspor ke JSON.")

async def step1_export():
    # ==================================================
    # SESUAIKAN DUA BULAN YANG INGIN DIBANDINGKAN
    # ==================================================
    await export_all_data("Juni", "2026", "2026-06-01T00:00:00Z", "2026-06-30T23:59:59Z")
    await export_all_data("Juli", "2026", "2026-07-01T00:00:00Z", "2026-07-31T23:59:59Z")

await step1_export()

### TAHAP 2: Agregasi Data Ekosistem & Laporan Perbandingan Gemini
Membaca seluruh file dari kedua bulan dan menyusun Laporan Perbandingan Komprehensif.

In [ ]:
def baca_json(filepath):
    if not os.path.exists(filepath):
        return []
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

def hitung_statistik_bulan(bulan, tahun):
    b_t = f"{bulan.lower()}{tahun}"
    csv_job = f"data_jobhistories_{b_t}.csv"
    if not os.path.exists(csv_job):
        return f"Data {bulan} {tahun} tidak ditemukan."
        
    df_job = pd.read_csv(csv_job)
    
    # ================== GAME & STATUS ANALYTICS ==================
    game_stats_text = ""
    if 'game' in df_job.columns and 'jobStatus' in df_job.columns and 'nc.total' in df_job.columns:
        status_counts = df_job.groupby(['game', 'jobStatus']).size().unstack(fill_value=0)
        df_completed_only = df_job[df_job['jobStatus'] == 'COMPLETED']
        rev_by_game = df_completed_only.groupby('game')['nc.total'].sum()
        
        game_stats_list = []
        for g in df_job['game'].unique():
            if pd.isna(g): continue
            comp = status_counts.loc[g, 'COMPLETED'] if 'COMPLETED' in status_counts.columns and g in status_counts.index else 0
            canc = status_counts.loc[g, 'CANCELED'] if 'CANCELED' in status_counts.columns and g in status_counts.index else 0
            rev = rev_by_game.get(g, 0)
            game_stats_list.append(f"{g}: {comp} Sukses, {canc} Batal | Pendapatan: {rev:,.2f} NC")
        game_stats_text = "\n      ".join(game_stats_list)

    # ================== GAMEMODE ANALYTICS ==================
    mode_stats_text = ""
    if 'gameMode' in df_job.columns:
        mode_counts = df_job['gameMode'].value_counts()
        mode_stats_text = ", ".join([f"{k}: {v} jobs" for k, v in mode_counts.items()])

    # ================== MARKET TYPE ANALYTICS ==================
    market_stats_text = ""
    if 'marketType' in df_job.columns:
        market_counts = df_job['marketType'].value_counts()
        market_stats_text = ", ".join([f"{k}: {v} jobs" for k, v in market_counts.items()])

    # Lanjutkan analisa utama HANYA untuk job yang COMPLETED
    if 'jobStatus' in df_job.columns:
        df_job = df_job[df_job['jobStatus'] == 'COMPLETED']
        
    total_jobs = len(df_job)
    total_revenue = df_job['nc.total'].sum() if 'nc.total' in df_job.columns else 0
    total_distance = df_job['distanceKm'].sum() if 'distanceKm' in df_job.columns else 0
    total_xp = df_job['xp.total'].sum() if 'xp.total' in df_job.columns else 0
    total_discount = df_job['discount.total'].sum() if 'discount.total' in df_job.columns else 0
    
    # ================== DURATION ANALYTICS ==================
    duration_text = ""
    if 'durationSeconds' in df_job.columns:
        total_seconds = df_job['durationSeconds'].sum()
        hours = int(total_seconds // 3600)
        minutes = int((total_seconds % 3600) // 60)
        duration_text = f"Total Waktu Mengemudi: {hours} Jam {minutes} Menit"
        
    # ================== VEHICLE STATUS ==================
    vehicle_status_text = "Tidak Ada Data Kepemilikan"
    if 'vehicleStatus' in df_job.columns:
        status_counts = df_job['vehicleStatus'].value_counts()
        vehicle_status_text = ", ".join([f"{k}: {v} jobs" for k, v in status_counts.items()])

    # ================== HARDCORE ANALYTICS ==================
    hardcore_text = "Tidak ada data hardcore."
    if 'isHardcore' in df_job.columns and 'hardcoreRating' in df_job.columns:
        df_hardcore = df_job[df_job['isHardcore'] == True]
        hc_count = len(df_hardcore)
        if hc_count > 0:
            avg_hc_rating = df_hardcore['hardcoreRating'].mean()
            hardcore_text = f"{hc_count} Job Hardcore diselesaikan (Rata-rata Rating: {avg_hc_rating:.2f} ⭐)"
    
    # ================== DAMAGE ==================
    total_damage_poin = 0
    for col in ['damage.vehicle', 'damage.trailer', 'damage.cargo']:
        if col in df_job.columns:
            total_damage_poin += df_job[col].sum()
    
    if total_jobs > 0:
        avg_dmg = total_damage_poin / total_jobs
        avg_damage_text = f"Akumulasi Persentase Kerusakan: {total_damage_poin:,.2f}% (Rata-rata {avg_dmg:,.2f}% kerusakan per pengiriman)"
    else:
        avg_damage_text = "Kerusakan: 0%"

    # ================== PENALTY BREAKDOWN ==================
    total_penalty = df_job['penalty.total'].sum() if 'penalty.total' in df_job.columns else 0
    p_vehicle = df_job['penalty.vehicle'].sum() if 'penalty.vehicle' in df_job.columns else 0
    p_trailer = df_job['penalty.trailer'].sum() if 'penalty.trailer' in df_job.columns else 0
    p_cargo = df_job['penalty.cargo'].sum() if 'penalty.cargo' in df_job.columns else 0
    p_speed = df_job['penalty.speed'].sum() if 'penalty.speed' in df_job.columns else 0
    p_dist = df_job['penalty.distance'].sum() if 'penalty.distance' in df_job.columns else 0
    
    penalty_breakdown = f"Kendaraan ({p_vehicle}), Trailer ({p_trailer}), Kargo ({p_cargo}), Kecepatan ({p_speed}), Jarak ({p_dist})"

    total_cost = df_job['ncCost.total'].sum() if 'ncCost.total' in df_job.columns else 0
    
    top_cargo = ""
    if 'cargoName' in df_job.columns and 'nc.total' in df_job.columns:
        top_5 = df_job.groupby('cargoName')['nc.total'].sum().sort_values(ascending=False).head(5)
        top_cargo = ", ".join([f"{k} ({v:,.0f} NC)" for k, v in top_5.items()])

    # ================== 2-6: EKOSISTEM ==================
    convoys = baca_json(f"data_convoylobby_{b_t}.json")
    coupons = baca_json(f"data_coupons_{b_t}.json")
    contracts = baca_json(f"data_contracthistories_{b_t}.json")
    ncevents = baca_json(f"data_ncevents_{b_t}.json")
    pointhistories = baca_json(f"data_pointhistories_{b_t}.json")
    
    total_claims = sum([len(c.get('driverClaims', [])) for c in coupons])
    total_contract_jobs = sum([c.get('completedContracts', 0) for c in contracts])
    total_contract_nc = sum([c.get('totalNCEarned', 0) for c in contracts])
    penalty_added = sum([p.get('points', 0) for p in pointhistories if p.get('type') == 'add'])
    penalty_paid = sum([p.get('points', 0) for p in pointhistories if p.get('type') == 'remove'])
    
    summary = f"""
    STATISTIK {bulan.upper()} {tahun}:
    
    [LOGISTIK & GAME]
    - Rincian Per Game:
      {game_stats_text}
    - Tren Platform Bermain: {mode_stats_text}
    - Tren Tipe Pasar (Market Type): {market_stats_text}
    - Total Pengiriman Sukses: {total_jobs} Jobs (Jarak: {total_distance:,.2f} Km)
    - {duration_text}
    - Total Pendapatan Kotor (NC): {total_revenue:,.2f} NC
    - Total Experience (XP) Didapat Driver: {total_xp:,.0f} XP
    - 5 Kargo Pencetak Pendapatan Terbesar: {top_cargo}
    - Tren Kepemilikan Truk: {vehicle_status_text}
    - Statistik Mode Hardcore: {hardcore_text}
    
    [RISIKO & PENGELUARAN]
    - {avg_damage_text}
    - Total Penalti Driver (Dari Job): {total_penalty:,.2f} Poin
      Rincian Penalti: {penalty_breakdown}
    - Total Biaya Operasional (Servis/Rental/Denda): {total_cost:,.2f} NC
    - Total Penghematan Driver (Diskon Asuransi/Nismara+): {total_discount:,.2f} NC
    
    [EKOSISTEM KOMUNITAS & MANAJEMEN]
    - Total Event Convoy (Mabar) Diselenggarakan: {len(convoys)} Convoy
    - Special Contracts Selesai: {total_contract_jobs} pengiriman khusus senilai {total_contract_nc:,.2f} NC dari {len(contracts)} kontrak aktif.
    - Event Bonus NC (NC Events): {len(ncevents)} event diselenggarakan.
    - Penebaran Kupon: {len(coupons)} kupon dirilis dan berhasil diklaim sebanyak {total_claims} kali oleh driver.
    - Pembayaran/Pelunasan Penalti: Tercatat {penalty_paid} poin penalti telah dilunasi oleh driver (remove), dari total {penalty_added} poin penalti baru yang masuk ke sistem (add).
    """
    return summary

def buat_laporan_komprehensif():
    summary_juni = hitung_statistik_bulan("Juni", "2026")
    summary_juli = hitung_statistik_bulan("Juli", "2026")
    
    print("📊 Seluruh data ekosistem dari DUA bulan berhasil dirangkum. Menyerahkan ke Gemini...")
    
    prompt = f"""
    Anda adalah Senior Data Analyst & Community Manager di Nismara Transport.
    Berikut adalah rangkuman data operasional murni beserta data ekosistem komunitas kami:
    
    {summary_juni}
    
    {summary_juli}
    
    TUGAS ANDA:
    Buatlah "Laporan Aktivitas Komprehensif Perusahaan (Juni vs Juli 2026)" yang ditujukan kepada Direktur Utama.
    Laporan harus tajam, profesional, menggunakan format bullet-points yang menarik, dan mencakup poin-poin perbandingan berikut:
    1. Perbandingan Kinerja Logistik per Game: Bandingkan tren peningkatan/penurunan volume dan pendapatan antara divisi ETS2 dan ATS. Sebutkan tren kepemilikan armada (Owned/Rental) dan Market Type (Freight/Cargo).
    2. Evaluasi Efisiensi & Keselamatan: Analisa tren platform bermain (SP vs TruckersMP). Bandingkan tingkat persentase kerusakan dan rincian poin penalti. Konteks Bisnis: Toleransi kerusakan di Singleplayer (SP) sangat ketat (10%), sementara TruckersMP (TMP) dilonggarkan hingga 20% karena resiko ditabrak pemain lain. Bahas efektivitas penggunaan diskon asuransi (penghematan).
    3. Mode Hardcore: Bandingkan antusiasme driver memainkan mode Hardcore antara dua bulan tersebut dan rata-rata ratingnya.
    4. Analisis Ekosistem & Komunitas: Evaluasi tren keaktifan anggota di event Convoy dan penyelesaian Special Contracts. Bandingkan efektivitas sistem penegakan kedisiplinan (rasio poin penalti baru vs poin dibayar/dilunasi).
    5. Kesimpulan & Rekomendasi Strategis: Berikan 2 langkah konkrit ke depan.
    
    PENTING: 
    - Semua nilai uang menggunakan "NC" (Nismara Coin). JANGAN MENGGUNAKAN SIMBOL USD ($).
    - Angka damage merupakan PERSENTASE KERUSAKAN, jangan disebut sebagai Poin/Point.
    - Penalti baru didapat secara otomatis dari sistem game, bukan manual dari manajer.
    - Bandingkan datanya secara eksplisit (Misal: "Pendapatan naik dari X menjadi Y").
    """
    
    response = model.generate_content(prompt)
    print("\n=======================================================")
    print("LAPORAN ANALISA KOMPREHENSIF GEMINI (PERBANDINGAN)")
    print("=======================================================\n")
    print(response.text)

buat_laporan_komprehensif()
